# Lecture 4 follow-along — Hello, Robot

Every code cell is **verbatim** from the slides, in slide order — the same cells you typed into `lec4_robot.py`. Run top to bottom; every printed number is derived on the matching slide before it runs.

## Build File 1 — `lec4_robot.py` (type along)
*NEW FILE · COMPLETE — `lec4_robot.py`: type it, **predict the printout**, run `python3 lec4_robot.py`*  
`→ lec4_robot.py`

In [ ]:
import numpy as np                        # the array engine, as in Lecture 3
DT = 0.1                                  # tick length [s] -> 10 Hz, like the sensor

def step(state, cmd, dt=DT, disturbance=0.03):   # one tick of physics, boxed
    x, y, yaw, d = state                  # unpack: four numbers get names
    u, r, w = cmd                         # surge, yaw-rate, heave
    return np.array([x + u*np.cos(yaw)*dt,        # planar motion follows heading
                     y + u*np.sin(yaw)*dt,
                     yaw + r*dt,                  # Euler on heading
                     max(0.0, d + (w + disturbance)*dt)])  # heavy trim + hard surface

print(step(np.array([0., 0., 0., 0.]), [0.5, 0.0, 0.2]))   # at rest; ask surge .5, heave .2

## Build — the mission runner
*ADD TO `lec4_robot.py` — below `step()`; run `python3 lec4_robot.py`*  
`→ lec4_robot.py`

In [ ]:
mission = [([0.0, 0.0, 0.25], 4.0),   # dive for 4 s
           ([0.6, 0.0, 0.0 ], 8.0),   # cruise straight 8 s
           ([0.4, 0.35, 0.0], 6.0),   # turning leg 6 s
           ([0.6, 0.0, 0.0 ], 6.0)]   # straight again
state = np.array([0., 0., 0., 0.])    # born at the surface, pointed along x
traj = [state]                        # the story so far — page one already written
for cmd, dur in mission:              # visit each phase of the plan
    for _ in range(int(dur/DT)):      # seconds -> ticks: 4 s at 10 Hz = 40 ticks
        state = step(state, cmd); traj.append(state)   # physics, then one more story row
T = np.array(traj)                    # the list collected it; the array computes on it
print(f"final: x={T[-1,0]:.2f}  y={T[-1,1]:.2f}  depth={T[-1,3]:.3f}")   # the receipt

## It moves — a gullible robot
*ADD TO `lec4_robot.py` — the plot block; run, then click `topview.png`*  
`→ lec4_robot.py`

In [ ]:
import matplotlib.pyplot as plt; tt = np.arange(T.shape[0]) * DT  # a time stamp per row
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.2, 2.0), width_ratios=[1, 2.2])  # 2 panels
a1.plot(T[:,0], T[:,1], lw=2, color='#0e7c86'); a1.set_aspect('equal')  # meter = meter
a1.set_xlabel('x [m]'); a1.set_ylabel('y [m]'); a1.set_title('top view')
a2.plot(tt, T[:,3], lw=2, color='#e07b00'); a2.invert_yaxis()   # flipped: down = deeper
a2.set_xlabel('time [s]'); a2.set_ylabel('depth [m]  (down = deeper)')
fig.subplots_adjust(bottom=0.28, wspace=0.32); plt.savefig("topview.png", dpi=110); plt.show()

## Commanding zero and hoping
*ADD TO `lec4_robot.py` — hold 1.0 m, w = 0, 30 s. Predict the ending; run, click `hold.png`*  
`→ lec4_robot.py`

In [ ]:
s = np.array([0., 0., 0., 1.0]); log = [1.0]   # start AT 1 m; log = the depth story
for _ in range(300):                           # 30 s of ticks, commanding w = 0
    s = step(s, [0.5, 0.0, 0.0]); log.append(s[3])   # tick the physics; record only depth
plt.figure(figsize=(8.2, 2.0)); plt.plot(np.arange(len(log))*DT, log, c='#e07b00')  # what happened
plt.axhline(1.0, ls='--', c='k'); plt.gca().invert_yaxis()  # the promise; down = deeper
plt.xlabel('time [s]'); plt.ylabel('depth [m]  (down = deeper)')
plt.subplots_adjust(bottom=0.28); plt.savefig("hold.png"); plt.show(); print(f"after 30 s: {log[-1]:.3f} m")

## Build — the rescue
*ADD TO `lec4_robot.py` — nine lines close the loop*  
`→ lec4_robot.py`

In [ ]:
rng = np.random.default_rng(7)             # Lecture 3's scramble machine
Kp, target = 0.8, 1.2                      # the dial and the goal
s = np.array([0., 0., 0., 0.])             # start at rest on the surface
zs = [0.0]                                 # the depth log — the story we plot next
for _ in range(400):                       # 40 s
    z = s[3] + rng.normal(0, 0.05)         # SENSE (the lie, as always)
    w = Kp * (target - z)                  # THINK (the law)
    w = max(-0.4, min(0.4, w))             # clamp to real thruster authority
    s = step(s, [0.5, 0.0, w]); zs.append(s[3])   # ACT

## The rescue, plotted
*ADD TO `lec4_robot.py` — run `python3 lec4_robot.py`, click `rescue.png`*  
`→ lec4_robot.py`

In [ ]:
plt.figure(figsize=(8.2, 2.0))                 # fresh canvas — a script reuses the last one
tt3 = np.arange(len(zs)) * DT                  # a time stamp per story entry
plt.plot(tt3, zs, lw=2, color='#0e7c86', label='P-controlled')   # what happened
plt.axhline(target, ls='--', c='k', lw=1)      # what we asked for
plt.gca().invert_yaxis(); plt.legend(loc='center right', fontsize=8)
plt.xlabel('time [s]'); plt.ylabel('depth [m]  (down = deeper)')
plt.subplots_adjust(bottom=0.28); plt.savefig("rescue.png", dpi=110); plt.show()

## Build — box the loop
*ADD TO `lec4_robot.py` — the rescue loop, boxed; run — **predict**: what should `boxed:` print?*  
`→ lec4_robot.py`

In [ ]:
def run_kp(Kp, N=None, T_end=40.0):        # dials: aggression, filter window, mission time
    rr = np.random.default_rng(7)          # same seed -> same lies -> fair comparisons
    ss = np.array([0., 0., 0., 0.]); zz = [0.0]; ww = [0.0]; win = []  # at rest; empty logs
    for _ in range(int(T_end/DT)):         # seconds -> ticks, now adjustable
        z = ss[3] + rr.normal(0, 0.05)     # SENSE (the lie, as always)
        if N:                              # filter dial ON: believe the last N readings
            win.append(z)                  # Lab 2's window, kept alive tick by tick
            if len(win) > N: win.pop(0)    # full window -> oldest reading leaves
            z = sum(win) / len(win)        # the mean is what the controller now sees
        w = max(-0.4, min(0.4, Kp*(1.2 - z)))    # THINK: the law, then the clamp
        ss = step(ss, [0.5, 0.0, w]); zz.append(ss[3]); ww.append(w)   # ACT + log
    return np.array(zz), np.array(ww)      # the story and the commands
print("boxed:", np.allclose(run_kp(0.8)[0], zs))  # the box retells the rescue, digit for digit

## Build — turn the dial yourself
*ADD TO `lec4_robot.py` — the four-dial overlay; run, then click `dials.png`*  
`→ lec4_robot.py`

In [ ]:
plt.figure(figsize=(8.2, 2.0))             # fresh canvas — a script reuses the last one
for Kp, c in zip([0.1, 0.4, 0.8, 3.0], ['#E69F00', '#56B4E9', '#009E73', '#D55E00']):  # four dials
    zzK, _ = run_kp(Kp)                    # same seed: every dial fights the same water
    plt.plot(np.arange(len(zzK))*DT, zzK, lw=1.6, color=c, label=f'Kp={Kp}')
plt.axhline(1.2, ls='--', c='k', lw=1); plt.legend(loc='lower right', fontsize=7, ncol=4)
plt.gca().invert_yaxis(); plt.xlabel('time [s]'); plt.ylabel('depth [m]  (down = deeper)')
plt.subplots_adjust(bottom=0.28); plt.savefig("dials.png", dpi=110); plt.show()

## The filter strikes back
*ADD TO `lec4_robot.py` — Lab 2's `moving_average`, unchanged*  
`→ lec4_robot.py`

In [ ]:
def moving_average(z, N):
    out = []                    # filtered result, tick by tick
    window = []                 # the last N readings — a moving bucket
    for zi in z:                # for each new reading...
        window.append(zi)       #   drop it in the bucket
        if len(window) > N:     #   bucket over-full?
            window.pop(0)       #   remove the OLDEST (index 0)
        out.append(sum(window) / len(window))   # the mean: the equation, wearing code
    return out                  # one calm number per tick, kept

## Three loops, three filters
*ADD TO `lec4_robot.py` — the comparison plot; run, then click `filters.png`*  
`→ lec4_robot.py`

In [ ]:
plt.figure(figsize=(8.2, 2.0))             # fresh canvas — a script reuses the last one
for N, c in zip([1, 10, 40], ['#56B4E9', '#009E73', '#D55E00']):  # three dials, one loop
    zzN, wwN = run_kp(0.8, N=None if N == 1 else N)    # N=1 means: believe the raw dots
    plt.plot(np.arange(len(zzN))*DT, zzN, lw=1.8, color=c, label=f'N={N}')
plt.axhline(1.2, ls='--', c='k', lw=1); plt.legend(loc='lower right', fontsize=7, ncol=3)
plt.gca().invert_yaxis(); plt.xlabel('time [s]'); plt.ylabel('depth [m]  (down = deeper)')
plt.subplots_adjust(bottom=0.28); plt.savefig("filters.png", dpi=110); plt.show()